<!-- course-header -->
# Lesson 08: Attention and the transformer

*Companion notebook for lesson 08. Read the lesson [online](https://ashutoshjoshi1.github.io/AI-ZerotoMastery/lessons/08-transformers.html) (it works well on a phone) or [offline](../lessons/08-transformers.html). Worked solutions are in `solutions/08-transformers-solutions.ipynb`; try each exercise first.*

<a href="https://colab.research.google.com/github/ashutoshjoshi1/AI-ZerotoMastery/blob/main/notebooks/08-transformers.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

The architecture behind every modern LLM, built up piece by piece: self-attention in NumPy, masking and multiple heads, the transformer block, then a small GPT you train yourself in PyTorch.

**Runs on:** The NumPy sections run anywhere. Training the tiny GPT takes minutes on a GPU and much longer on a CPU.

**How to use it:** read a section of the lesson, then run the matching cells here, in order. Each exercise has a cell titled *Hands-on* with the same name as in the lesson. Change numbers, break things on purpose, and note what happens in your learning log. Cells that need an API key, a GPU or a download say so in their first line.

## Setup

Install what this lesson needs. Skip this cell if your environment from lesson 00 already has these packages.

In [ ]:
%pip install -q numpy torch

## Self-attention, step by step

That one line is the heart of every LLM. Here it is in NumPy:

In [ ]:
import numpy as np

rng = np.random.default_rng(0)

def softmax(z, axis=-1):
    z = z - z.max(axis=axis, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)

T, d_model, d_k = 4, 8, 8                    # 4 tokens, 8-dim vectors
X = rng.normal(size=(T, d_model))            # stand-in for 4 token embeddings
Wq, Wk, Wv = (rng.normal(0, d_model ** -0.5, (d_model, d_k)) for _ in range(3))

Q, K, V = X @ Wq, X @ Wk, X @ Wv             # each (4, 8)
scores = Q @ K.T / np.sqrt(d_k)              # (4, 4): token i's score for token j
weights = softmax(scores)                    # each row sums to 1
out = weights @ V                            # (4, 8): a new vector for each token
print(weights.round(2))

```text
[[0.34 0.13 0.33 0.2 ]
 [0.78 0.   0.05 0.17]
 [0.25 0.28 0.29 0.18]
 [0.22 0.28 0.21 0.29]]
```

### Why divide by √d_k?

The dot product of two random vectors with $d$ independent unit-variance components has standard deviation $\sqrt{d}$. With large $d$, scores get huge, softmax saturates into a near one-hot distribution, and its gradients vanish. Dividing by $\sqrt{d_k}$ keeps scores at unit scale whatever the dimension.

In [ ]:
for d in [4, 64, 1024]:
    q, k = rng.normal(size=d), rng.normal(size=(1000, d))
    print(d, (k @ q).std().round(1), (k @ q / np.sqrt(d)).std().round(1))
# 4 1.8 0.9    64 8.6 1.1    1024 31.5 1.0

## Masking, heads and positions

### Causal masking: no peeking at the future

A language model predicts the next token, so position $i$ must not see positions after it. Set those scores to $-\infty$ before softmax; they become exactly zero weight.

In [ ]:
mask = np.triu(np.ones((T, T), dtype=bool), k=1)   # True above the diagonal = the future
weights = softmax(np.where(mask, -np.inf, scores))
print(weights.round(2))
# [[1.   0.   0.   0.  ]     token 0 can only see itself
#  [1.   0.   0.   0.  ]
#  [0.31 0.34 0.35 0.  ]
#  [0.22 0.28 0.21 0.29]]    the last token sees everything

### Multi-head attention

One attention pattern per layer is limiting: a token may need to track its syntactic head, the topic, and a matching quotation mark all at once. **Multi-head attention** splits the model dimension into $h$ smaller heads, runs attention independently in each, concatenates the results and mixes them with an output projection $W_O$. Same cost, many different "views".

In [ ]:
def multi_head(X, Wq, Wk, Wv, Wo, n_heads, causal=True):
    T, d_model = X.shape
    d_head = d_model // n_heads
    # (T, d_model) -> (n_heads, T, d_head)
    split = lambda M: M.reshape(T, n_heads, d_head).transpose(1, 0, 2)
    Q, K, V = split(X @ Wq), split(X @ Wk), split(X @ Wv)
    s = Q @ K.transpose(0, 2, 1) / np.sqrt(d_head)          # (n_heads, T, T)
    if causal:
        s = np.where(np.triu(np.ones((T, T), dtype=bool), 1), -np.inf, s)
    out = softmax(s) @ V                                      # (n_heads, T, d_head)
    out = out.transpose(1, 0, 2).reshape(T, d_model)          # concatenate heads
    return out @ Wo

d_model, H = 16, 4
X = rng.normal(size=(6, d_model))
Ws = [rng.normal(0, d_model ** -0.5, (d_model, d_model)) for _ in range(4)]
print(multi_head(X, *Ws, n_heads=H).shape)                    # (6, 16)

## Build a tiny GPT

### Data

In [ ]:
# Downloads a model or data on first run
import os, urllib.request
URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
if not os.path.exists("input.txt"):
    urllib.request.urlretrieve(URL, "input.txt")
print(open("input.txt", encoding="utf-8").read()[:200])

In [ ]:
# Needs PyTorch; a GPU runtime is much faster
import math, torch, torch.nn as nn, torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)

In [ ]:
# Downloads a model or data on first run
# (reads input.txt, which the download cell under Data saves)
# wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
text = open("input.txt", encoding="utf-8").read()
chars = sorted(set(text))
vocab_size = len(chars)                                   # 65 for Tiny Shakespeare
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[i] for i in ids)

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]

block_size, batch_size = 128, 64                           # context length, sequences per batch

def get_batch(split):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - block_size - 1, (batch_size,))
    x = torch.stack([d[i : i + block_size] for i in ix])          # (B, T)
    y = torch.stack([d[i + 1 : i + block_size + 1] for i in ix])  # targets: shifted by one
    return x.to(device), y.to(device)

### Model

In [ ]:
# Needs PyTorch; a GPU runtime is much faster
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)        # W_Q, W_K, W_V in one matrix
        self.proj = nn.Linear(d_model, d_model)           # W_O
        self.attn_drop = nn.Dropout(dropout)
        self.resid_drop = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        # (B, T, C) -> (B, n_heads, T, head_dim)
        q, k, v = (t.view(B, T, self.n_heads, C // self.n_heads).transpose(1, 2) for t in (q, k, v))
        att = (q @ k.transpose(-2, -1)) / math.sqrt(k.size(-1))           # (B, H, T, T)
        mask = torch.triu(torch.ones(T, T, dtype=torch.bool, device=x.device), diagonal=1)
        att = att.masked_fill(mask, float("-inf")).softmax(dim=-1)
        y = self.attn_drop(att) @ v                                        # (B, H, T, head_dim)
        y = y.transpose(1, 2).contiguous().view(B, T, C)                   # re-join heads
        return self.resid_drop(self.proj(y))
        # Production code replaces the four attention lines with
        # F.scaled_dot_product_attention(q, k, v, is_causal=True), which uses FlashAttention kernels.


class Block(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, 4 * d_model), nn.GELU(),
            nn.Linear(4 * d_model, d_model), nn.Dropout(dropout),
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))       # tokens talk to each other
        x = x + self.mlp(self.ln2(x))        # each token thinks on its own
        return x


class TinyGPT(nn.Module):
    def __init__(self, vocab_size, block_size, d_model=256, n_heads=4, n_layers=4, dropout=0.1):
        super().__init__()
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(block_size, d_model)          # learned absolute positions
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([Block(d_model, n_heads, dropout) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight                    # weight tying: input and output share embeddings
        self.apply(self._init)

    def _init(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)         # small init keeps the initial loss near ln(vocab)
        if isinstance(m, nn.Linear) and m.bias is not None:
            nn.init.zeros_(m.bias)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.drop(self.tok_emb(idx) + self.pos_emb(pos))      # (B, T, C)
        for block in self.blocks:
            x = block(x)
        logits = self.head(self.ln_f(x))                          # (B, T, vocab)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -self.block_size:])          # crop to the context window
            logits = logits[:, -1, :] / temperature               # only the last position matters
            if top_k is not None:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = float("-inf")
            next_id = torch.multinomial(F.softmax(logits, dim=-1), num_samples=1)
            idx = torch.cat([idx, next_id], dim=1)
        return idx

### Train and sample

In [ ]:
# Needs a GPU runtime
# (it also runs on a CPU, but 5,000 steps take much longer there)
model = TinyGPT(vocab_size, block_size).to(device)
print(sum(p.numel() for p in model.parameters()) / 1e6, "M parameters")   # about 3.2M
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.1)

@torch.no_grad()
def estimate_loss(iters=50):
    model.eval()
    out = {}
    for split in ["train", "val"]:
        out[split] = sum(model(*get_batch(split))[1].item() for _ in range(iters)) / iters
    model.train()
    return out

for step in range(5001):
    if step % 500 == 0:
        print(step, estimate_loss())          # step 0 should show about ln(65) = 4.17
    x, y = get_batch("train")
    _, loss = model(x, y)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()

start = torch.tensor([encode("ROMEO:")], device=device)
print(decode(model.generate(start, 400, temperature=0.8, top_k=40)[0].tolist()))

### Hands-on: experiments on your GPT

1. Remove the causal mask and retrain for a few hundred steps. The training loss drops suspiciously fast. Why, and what happens to generation?
2. Delete the position embedding. How does the validation loss and the sample quality change?
3. Double `n_layers` to 8, then separately double `d_model` to 512. Record parameters, validation loss and time per step. Which gave more improvement per parameter?
4. Train on a different text: your own notes, a public-domain book, or Python source code.

In [ ]:
# Your code here

## Generation, the KV cache and the cost of attention

### Attention is quadratic in sequence length

## Check yourself

Take the quiz at the end of [the lesson page](../lessons/08-transformers.html). Then close everything and write three or four sentences in your learning log about what you learned, from memory.